<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo"  />
    </a>
</p>


# **Multimodal Similarity Fusion and Retrieval Ranking**


Estimated time needed: **45** minutes


## Introduction

Earlier, you built the multimodal retrieval backbone by constructing vector indexes for restaurant articles and food images. You then operationalized these indexes by performing similarity-based retrieval with optional metadata filtering. At that stage, ranking was performed independently within each modality.

In this lab, you take the next step toward a production-grade multimodal retrieval system. Real-world RAG pipelines rarely rely on a single evidence source. Instead, they retrieve candidates from multiple modalities and **fuse** them into a unified ranking that better reflects overall relevance.

To enable this, the lab introduces **cross-modal score normalization and fusion**. Because similarity scores produced by different embedding models (for example, Sentence-Transformers versus CLIP) are not directly comparable, they must first be calibrated onto a common scale. You'll then apply a weighted fusion strategy to combine evidence from text and image retrieval into a single ranked list.

You'll design and implement a practical multimodal fusion pipeline that mirrors real production systems. This includes retrieving candidates from both modalities, normalizing their scores, applying weighted fusion, and optionally enforcing metadata constraints to support controllable, precision-oriented retrieval.

By the end of the lab, you will have a unified multimodal ranking pipeline capable of intelligently prioritizing heterogeneous evidence sources.


## Objectives

After completing this lab, you will be able to:

- Compute cosine-based similarity scores for both text and image retrieval pipelines  
- Normalize similarity scores to enable fair comparison across embedding spaces  
- Implement weighted multimodal fusion to produce a unified ranked result set  
- Apply metadata constraints to support controllable, constraint-aware reranking  
- Analyze how fusion weights and filtering strategies affect retrieval behavior
- Build a production-style multimodal retrieval workflow that integrates heterogeneous evidence sources


## Set up of the environment 

In this lab, you will produce a **unified ranked list** by combining evidence from both the article and image indexes. We begin by importing the core libraries for numerical computation, model inference, and vector database access.

All required packages were installed in Lesson 1, so this step verifies that the runtime environment is ready.


In [1]:
# ================================
# Import environment
# (Dependencies were installed in Lesson 1)
# ================================

# Standard library
import os
from pathlib import Path

# Third-party
import numpy as np
import torch
from PIL import Image
from langchain_chroma import Chroma
from sentence_transformers import SentenceTransformer
from transformers import CLIPModel, CLIPProcessor

print("✅ Environment ready")


✅ Environment ready


## Verify vector databases

Before implementing multimodal fusion, you'll confirm that the Chroma databases created earlier are available and populated.

This validation step prevents downstream errors (for example, empty retrieval results) and ensures that this lab operates on the same persisted indexes used in earlier labs.


In [2]:
# ================================
# Verify vector database
# ================================

DB_DIR = str((Path.home() / "chroma_multimodal").resolve())

if not os.path.isdir(DB_DIR):
    raise RuntimeError(
        f"Vector database directory not found: '{DB_DIR}'. "
        "Please run Lesson 1 (Multimodal Vector Index Construction) first."
    )

article_db = Chroma(collection_name="restaurant_articles", persist_directory=DB_DIR)
image_db   = Chroma(collection_name="food_images",          persist_directory=DB_DIR)

n_articles = article_db._collection.count()
n_images   = image_db._collection.count()

if n_articles <= 0 or n_images <= 0:
    raise RuntimeError(
        "One or more collections are empty. Please rerun Lesson 1 to rebuild the index."
    )

print(f"✅ Article vectors: {n_articles}")
print(f"✅ Image vectors:   {n_images}")

✅ Article vectors: 95
✅ Image vectors:   109


## Initialize embedding models

To retrieve across modalities, you need to embed queries into the **same vector spaces** used when building the indexes:

- **Text embeddings (384-d)** for article retrieval (Sentence-Transformers)  
- **CLIP embeddings (512-d)** for image retrieval  
  - Earlier, you focused on **image → image**  
  - This lab adds **text → image** retrieval using CLIP’s text encoder  

All embeddings are **L2-normalized**, so cosine similarity is stable and comparable across modalities.

Because different models produce scores on different scales, you will **normalize similarity scores** later before fusing results across modalities.


In [3]:
# ================================
# Initialize embedding models
# ================================

# ---- Text embedding model (384-d) ----
text_model = SentenceTransformer("all-MiniLM-L6-v2")

def embed_texts(texts, batch_size=64):
    return text_model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=False,
        normalize_embeddings=True,  # cosine-ready
    ).astype(np.float32)

print("✅ Text embedder ready")


# ---- Image embedding model: SigLIP ----
from transformers import AutoProcessor, AutoModel
device = "cuda" if torch.cuda.is_available() else "cpu"

siglip_name = "google/siglip-base-patch16-224"

siglip_processor = AutoProcessor.from_pretrained(siglip_name)
siglip_model = AutoModel.from_pretrained(siglip_name).to(device)

siglip_model.eval()


@torch.no_grad()
def embed_images(paths, batch_size=16):
    vecs = []

    for i in range(0, len(paths), batch_size):
        batch = paths[i:i + batch_size]

        imgs = [
            Image.open(p).convert("RGB")
            for p in batch
        ]

        inputs = siglip_processor(
            images=imgs,
            return_tensors="pt"
        )

        inputs = {
            k: v.to(device)
            for k, v in inputs.items()
        }

        # Get SigLIP image representations
        outputs = siglip_model.get_image_features(**inputs)

        # Depending on Transformers version, outputs may be
        # a tensor OR BaseModelOutputWithPooling
        if isinstance(outputs, torch.Tensor):
            feats = outputs
        elif hasattr(outputs, "pooler_output"):
            feats = outputs.pooler_output
        elif hasattr(outputs, "last_hidden_state"):
            feats = outputs.last_hidden_state[:, 0]
        else:
            raise TypeError(
                f"Unexpected SigLIP output type: {type(outputs)}"
            )

        # Normalize for cosine similarity
        feats = torch.nn.functional.normalize(
            feats,
            p=2,
            dim=-1
        )

        vecs.append(
            feats.cpu().numpy().astype(np.float32)
        )

    return np.vstack(vecs)


print("✅ SigLIP image embedder ready")

/home/zahra/.pyenv/versions/3.10.8/lib/python3.10/site-packages/torch/cuda/__init__.py:188: UserWarning: CUDA initialization: The NVIDIA driver on your system is too old (found version 12080). Please update your GPU driver by downloading and installing a new version from the URL: http://www.nvidia.com/Download/index.aspx Alternatively, go to: https://pytorch.org to install a PyTorch version that has been compiled with your version of the CUDA driver. (Triggered internally at /__w/pytorch/pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  return torch._C._cuda_getDeviceCount() > 0


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

✅ Text embedder ready


[transformers] Model config: bos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49406. This may result in unexpected behavior.
[transformers] Model config: eos_token_id must be `None` or an integer within the vocabulary (between 0 and 31999), got 49407. This may result in unexpected behavior.


Loading weights:   0%|          | 0/408 [00:00<?, ?it/s]

✅ SigLIP image embedder ready


## Prepare utility functions

Chroma returns nested result structures (lists-of-lists) and **distance**-based scores. To support clean fusion logic, you will implement helper utilities that:

- Unwrap retrieval outputs  
- Convert cosine-style distances into similarity scores: `similarity = 1 - distance`
- Normalize similarity scores to `[0, 1]` for fusion  

These utilities ensure that cross-modal comparisons are meaningful and consistent.


In [4]:
# ================================
# Utilities
# ================================

def _unwrap(res: dict):
    """Chroma returns lists-of-lists; unwrap the first query."""
    ids   = res.get("ids", [[]])[0]
    docs  = res.get("documents", [[]])[0]
    metas = res.get("metadatas", [[]])[0]
    dists = res.get("distances", [[]])[0]
    return ids, docs, metas, dists

def _to_similarity(dists, metric: str = "auto"):
    """
    Convert Chroma distances to a 'higher-is-better' similarity.
    
    - For cosine distance (range ~[0, 2]):  sim = 1 - dist
    - For L2 / Euclidean:                  sim = 1 / (1 + dist)
    """
    d = np.array(dists, dtype=np.float32)
    
    if metric == "cosine" or (metric == "auto" and d.max() <= 2.0 and d.min() >= 0.0):
        # cosine distance → cosine similarity
        return 1.0 - d
    else:
        # L2 or unknown → inverse distance (always positive)
        return 1.0 / (1.0 + d)

def _minmax(x):
    """Min-max normalize to [0, 1] with safe handling for constant arrays."""
    x = np.array(x, dtype=np.float32)
    if x.size == 0:
        return x
    lo, hi = float(x.min()), float(x.max())
    if abs(hi - lo) < 1e-8:
        return np.ones_like(x)  # all equal -> treat as same confidence
    return (x - lo) / (hi - lo)

def print_hits(ids, docs, metas, scores, title: str, max_chars: int = 140):
    print(f"\n=== {title} ===")
    for i in range(len(ids)):
        meta = metas[i] if i < len(metas) else {}
        score = float(scores[i]) if i < len(scores) else None

        snippet = (docs[i] or "").replace("\n", " ").strip()
        if len(snippet) > max_chars:
            snippet = snippet[:max_chars].rstrip() + "..."

        cuisine = meta.get("cuisine", "N/A") if isinstance(meta, dict) else "N/A"
        location = meta.get("location", "N/A") if isinstance(meta, dict) else "N/A"
        doc_id = meta.get("doc_id", ids[i]) if isinstance(meta, dict) else ids[i]
        source = meta.get("source", "N/A") if isinstance(meta, dict) else "N/A"

        print(f"[{i+1}] id={doc_id} | cuisine={cuisine} | location={location} | source={source} | score={score:.4f}")
        print(f"{snippet}")

## Define Retrieval functions (text → articles, text → images)

Next, define two retrieval functions that accept the **same user query** but search **different vector indexes**. The query is encoded into:

- **MiniLM text space (384-d)** for text→article retrieval  
- **CLIP multimodal space (512-d)** for text→image retrieval  

Each embedding retrieves top-k candidates from its corresponding collection. To support downstream fusion, both functions return a **standardized output format** (IDs, documents, metadata, and similarity scores), enabling cross-modal normalization and unified ranking. Both functions optionally accept metadata filters via `where=...`, enabling **constraint-aware retrieval** (for example, restricting by `source`, `location`, or `cuisine`).


In [5]:
# ================================
# Retrieval functions
# ================================

def retrieve_articles(query: str, k: int = 5, where: dict | None = None):
    q_vec = embed_texts([query])[0]  # 384-d
    res = article_db._collection.query(
        query_embeddings=[q_vec.tolist()],
        n_results=k,
        where=where,
        include=["documents", "metadatas", "distances"],
    )
    ids, docs, metas, dists = _unwrap(res)
    sims = _to_similarity(dists)
    return ids, docs, metas, sims



In [6]:
@torch.no_grad()
def embed_text_siglip(texts, batch_size: int = 32):
    """Embed text with SigLIP so it lives in the same space as the image vectors."""
    all_vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        inputs = siglip_processor(
            text=batch,
            padding="max_length",
            truncation=True,
            max_length=64,          # SigLIP default
            return_tensors="pt",
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}

        outputs = siglip_model.get_text_features(**inputs)

        if isinstance(outputs, torch.Tensor):
            feats = outputs
        elif hasattr(outputs, "pooler_output"):
            feats = outputs.pooler_output
        else:
            raise TypeError(f"Unexpected SigLIP text output: {type(outputs)}")

        feats = torch.nn.functional.normalize(feats, p=2, dim=-1)
        all_vecs.append(feats.cpu().numpy().astype(np.float32))

    return np.vstack(all_vecs)


def retrieve_images_by_text(query: str, k: int = 5, where: dict | None = None):
    q_vec = embed_text_siglip([query])[0]
    res = image_db._collection.query(
        query_embeddings=[q_vec.tolist()],
        n_results=k,
        where=where,
        include=["documents", "metadatas", "distances"],
    )
    ids, docs, metas, dists = _unwrap(res)
    sims = _to_similarity(dists)          # auto
    return ids, docs, metas, sims


print("✅ retrieve_images_by_text ready (SigLIP cross-modal)")

✅ retrieve_images_by_text ready (SigLIP cross-modal)


## Implement multimodal fusion and reranking

You will now combine article and image results into a **single ranked list**. 

The fusion process:

1. Retrieves top-k candidates from each modality  
2. Normalizes similarity scores within each modality  
3. Computes a weighted fused score:
   $$
   s_\text{fused} = w_\text{text} \cdot \hat{s}_\text{text} + w_\text{img} \cdot \hat{s}_\text{img}
   $$
4. Produces a single unified ranked list  

This fusion step enables cross-modal comparison and ranking. By adjusting modality weights and metadata filters, you can control the relative influence of text and image evidence, mirroring the production multimodal RAG systems.


In [7]:
# ================================
# Multimodal fusion
# ================================

def fuse_rank(
    query: str,
    k_text: int = 5,
    k_img: int = 5,
    w_text: float = 0.6,
    w_img: float = 0.4,
    where_text: dict | None = None,
    where_img: dict | None = None,
    top_n: int = 5
):
    # Retrieve per modality
    t_ids, t_docs, t_metas, t_sims = retrieve_articles(query, k=k_text, where=where_text)
    i_ids, i_docs, i_metas, i_sims = retrieve_images_by_text(query, k=k_img, where=where_img)

    # Normalize within modality
    t_norm = _minmax(t_sims)
    i_norm = _minmax(i_sims)

    # Build one mixed candidate list with fused scores
    rows = []
    for j in range(len(t_ids)):
        rows.append({
            "modality": "article",
            "id": t_metas[j].get("doc_id", t_ids[j]) if isinstance(t_metas[j], dict) else t_ids[j],
            "cuisine": t_metas[j].get("cuisine", "N/A") if isinstance(t_metas[j], dict) else "N/A",
            "location": t_metas[j].get("location", "N/A") if isinstance(t_metas[j], dict) else "N/A",
            "source": t_metas[j].get("source", "N/A") if isinstance(t_metas[j], dict) else "N/A",
            "text_score": float(t_norm[j]),
            "img_score": 0.0,
            "fused": float(w_text * t_norm[j]),
            "snippet": (t_docs[j] or "").replace("\n", " ").strip(),
        })

    for j in range(len(i_ids)):
        rows.append({
            "modality": "image",
            "id": i_metas[j].get("doc_id", i_ids[j]) if isinstance(i_metas[j], dict) else i_ids[j],
            "cuisine": i_metas[j].get("cuisine", "N/A") if isinstance(i_metas[j], dict) else "N/A",
            "location": i_metas[j].get("location", "N/A") if isinstance(i_metas[j], dict) else "N/A",
            "source": i_metas[j].get("source", "N/A") if isinstance(i_metas[j], dict) else "N/A",
            "text_score": 0.0,
            "img_score": float(i_norm[j]),
            "fused": float(w_img * i_norm[j]),
            "snippet": (i_docs[j] or "").replace("\n", " ").strip(),
        })

    # Sort by fused score (desc rerank)
    rows.sort(key=lambda r: r["fused"], reverse=True)
    
    # if top_n not specified, return full pool (k_text + k_img)
    if top_n is None:
        return rows

    top_n = max(0, min(int(top_n), len(rows)))
    return rows[:top_n]

def print_fused(rows, title: str, max_chars: int = 90):
    print(f"\n=== {title} ===")
    for idx, r in enumerate(rows, start=1):
        snippet = r["snippet"]
        if len(snippet) > max_chars:
            snippet = snippet[:max_chars].rstrip() + "..."
        print(
            f"[{idx}] {r['modality']} | id={r['id']} | cuisine={r['cuisine']} | "
            f"location={r['location']} | fused={r['fused']:.4f} "
            f"(text={r['text_score']:.4f}, img={r['img_score']:.4f})"
        )
        print(snippet)

## Demo 1 — Multimodal fusion (no filters)

Begin with the baseline scenario. The query retrieves candidates from both modalities and fuses them using default weights.

- **Article database:** text → text similarity  
- **Image database:** text → image similarity  

This demo demonstrates the full multimodal pipeline without any metadata constraints. 


In [8]:
# ================================
# Demo 1 — Multimodal fusion (no filters)
# ================================

q = "cozy noodles with warm atmosphere"

rows = fuse_rank(
    q,
    k_text=5,
    k_img=5,
    w_text=0.6,
    w_img=0.4,
    where_text=None,
    where_img=None,
    top_n=5
)

print_fused(rows, title="Demo 1 — Multimodal fusion (no filters)")
print("✅ Demo 1 complete")


=== Demo 1 — Multimodal fusion (no filters) ===
[1] article | id=rest_47 | cuisine=Ramen | location=Koreatown | fused=0.6000 (text=1.0000, img=0.0000)
Restaurant: The Neon Noodle Cuisine: Ramen Location: Koreatown
[2] image | id=img_100 | cuisine=Chinese (Szechuan) | location=N/A | fused=0.4000 (text=0.0000, img=1.0000)
Kung Pao Chicken
[3] article | id=rest_64 | cuisine= | location=Sacramento | fused=0.3752 (text=0.6253, img=0.0000)
Restaurant: The Neon Noodle Cuisine:  Location: Sacramento
[4] image | id=img_87 | cuisine=Mexican | location=N/A | fused=0.2242 (text=0.0000, img=0.5605)
Chili Lime Grilled Corn
[5] article | id=rest_4 | cuisine=Sichuan street food | location=San Gabriel Valley | fused=0.2188 (text=0.3647, img=0.0000)
Restaurant: The Neon Noodle Bar Cuisine: Sichuan street food Location: San Gabriel Valley
✅ Demo 1 complete


## Demo 2 — Multimodal fusion (metadata filters)

In production, you often want semantic relevance and constraints (e.g., “only Pasadena” or “only Italian”).

Here, you'll apply:
- A filter on the article database (for example, location)
- A filter on the image database (for example, source, cuisine)

If a filter is too strict, you may see fewer results. Observe how filtering affects the candidate pool and the final fused ranking.


In [9]:
# ================================
# Demo 2 — Multimodal fusion (metadata filters)
# ================================

q = "handmade pasta and romantic dinner"

where_articles = {"location": "Pasadena"}   # change to any location present in your dataset
where_images   = {"cuisine": "French"} # optional

rows = fuse_rank(
    q,
    k_text=5,
    k_img=5,
    w_text=0.6,
    w_img=0.4,
    where_text=where_articles,
    where_img=where_images,
    top_n=5
)

if len(rows) == 0:
    print("⚠️ No results found. Try relaxing filters (location/cuisine/source).")
else:
    print_fused(rows, title="Demo 2 — Multimodal fusion (metadata filters)")

print("✅ Demo 2 complete")


=== Demo 2 — Multimodal fusion (metadata filters) ===
[1] article | id=rest_3 | cuisine=French-Mediterranean | location=Pasadena | fused=0.6000 (text=1.0000, img=0.0000)
Restaurant: Velvet & Vine Cuisine: French-Mediterranean Location: Pasadena
[2] image | id=img_91 | cuisine=French | location=N/A | fused=0.4000 (text=0.0000, img=1.0000)
Herbed Quiche
[3] article | id=rest_55 | cuisine=refined afternoon tea service alongside modern Californian salads | location=Pasadena | fused=0.3921 (text=0.6536, img=0.0000)
Restaurant: The Green Gable Cuisine: refined afternoon tea service alongside modern Califo...
[4] image | id=img_94 | cuisine=French | location=N/A | fused=0.3829 (text=0.0000, img=0.9573)
Duck Confit (Simple Version)
[5] image | id=img_26 | cuisine=French | location=N/A | fused=0.1453 (text=0.0000, img=0.3632)
Ratatouille
✅ Demo 2 complete


## Demo 3 — Weight tuning (reranking behavior)

Finally, you'll explore how fusion weights influence ranking behavior. By adjusting the fusion weights, you can control which modality dominates the final ranking:

- Higher `w_text` → more article-heavy ranking
- Higher `w_img` → more image-heavy ranking

Try changing weights and observe how the top results shift.


In [14]:
# ================================
# Demo 3 — Weight tuning
# ================================

q = "pasta with tomato sauce"

# TODO:
# 1. Run fusion ranking with a text-heavy setting and print results (use title="Demo 3A — Text-heavy fusion (w_text=0.8, w_img=0.2)")
rows = fuse_rank(
    q,
    k_text=5,
    k_img=5,
    w_text=0.8,
    w_img=0.2,
    where_text=None,
    where_img=None,
    top_n=5
)
if len(rows) == 0:
    print("⚠️ No results found. Try relaxing filters (location/cuisine/source).")
else:
    print_fused(rows, title="Demo 2 — Multimodal fusion (text-heavy setting and print results)")

# 2. Run fusion ranking with an image-heavy setting and print results (use title="Demo 3B — Image-heavy fusion (w_text=0.3, w_img=0.7)")
rows = fuse_rank(
    q,
    k_text=5,
    k_img=5,
    w_text=0.2,
    w_img=0.8,
    where_text=None,
    where_img=None,
    top_n=5
)
if len(rows) == 0:
    print("⚠️ No results found. Try relaxing filters (location/cuisine/source).")
else:
    print_fused(rows, title="Demo 2 — Multimodal fusion (image-heavy setting)")

# 3. Select 5 results from each modality, but only show the top 5 fused results

# your code here

print("✅ Demo 3 complete")
print("🎉 Multimodal Similarity Fusion and Retrieval Ranking COMPLETE")



=== Demo 2 — Multimodal fusion (text-heavy setting and print results) ===
[1] article | id=rest_70 | cuisine=French-Italian | location=Napa | fused=0.8000 (text=1.0000, img=0.0000)
Restaurant: Velvet & Vine Cuisine: French-Italian Location: Napa
[2] article | id=rest_6 | cuisine=coastal Italian | location=Manhattan Beach | fused=0.5948 (text=0.7435, img=0.0000)
Restaurant: Salt & Sand Cuisine: coastal Italian Location: Manhattan Beach
[3] article | id=rest_64 | cuisine= | location=Sacramento | fused=0.4704 (text=0.5880, img=0.0000)
Restaurant: The Neon Noodle Cuisine:  Location: Sacramento
[4] article | id=rest_24 | cuisine=Japanese | location=Sawtelle | fused=0.3428 (text=0.4285, img=0.0000)
Restaurant: Miso Hungry Cuisine: Japanese Location: Sawtelle
[5] image | id=img_83 | cuisine=American | location=N/A | fused=0.2000 (text=0.0000, img=1.0000)
Salisbury Steak with Mushroom Gravy

=== Demo 2 — Multimodal fusion (image-heavy setting) ===
[1] image | id=img_83 | cuisine=American | lo

In [11]:
# 1. Raw image retrieval for the same query
ids, docs, metas, sims = retrieve_images_by_text(
    "fresh sushi and minimalist presentation", k=10
)
print_hits(ids, docs, metas, sims, title="Raw SigLIP text→image hits")


=== Raw SigLIP text→image hits ===
[1] id=img_53 | cuisine=American | location=N/A | source=recipe_image | score=-0.8902
Lemon Ricotta Pancakes
[2] id=img_14 | cuisine=Italian | location=N/A | source=recipe_image | score=-0.9025
Roasted Tomato and Basil Pasta
[3] id=img_40 | cuisine=American | location=N/A | source=recipe_image | score=-0.9106
Garlic Butter Shrimp
[4] id=img_5 | cuisine=American | location=N/A | source=recipe_image | score=-0.9185
Distance-Glazed Salmon with Dill
[5] id=img_11 | cuisine=Indian | location=N/A | source=recipe_image | score=-0.9228
Lentil Dal with Cumin and Turmeric
[6] id=img_86 | cuisine=Mexican | location=N/A | source=recipe_image | score=-0.9321
Shredded Beef Tostadas
[7] id=img_39 | cuisine=Italian | location=N/A | source=recipe_image | score=-0.9324
Pumpkin Risotto
[8] id=img_66 | cuisine=Italian | location=N/A | source=recipe_image | score=-0.9420
Pesto Gnocchi with Cherry Tomatoes
[9] id=img_8 | cuisine=Middle Eastern | location=N/A | source=reci

In [12]:
# 2. What cuisines actually exist in the image collection?
from collections import Counter

sample = image_db._collection.get(include=["metadatas"], limit=1000)
cuisines = [m.get("cuisine") for m in sample["metadatas"] if m and m.get("cuisine")]
print("Top cuisines in image collection:")
print(Counter(cuisines).most_common(20))
print(f"\nTotal images sampled: {len(sample['metadatas'])}")

Top cuisines in image collection:
[('American', 20), ('Italian', 16), ('Indian', 11), ('Thai', 6), ('Chinese', 5), ('Mediterranean', 5), ('Mexican', 4), ('Japanese', 4), ('French', 4), ('Moroccan', 4), ('Greek', 3), ('Middle Eastern', 3), ('Caribbean', 3), ('Indonesian', 3), ('Peruvian', 2), ('Chinese (Szechuan)', 2), ('Chinese (Cantonese)', 2), ('Cuban', 1), ('Korean', 1), ('Spanish', 1)]

Total images sampled: 109


In [13]:
# Check the distance metric of the image collection
print("Image collection metadata:")
print(image_db._collection.metadata)

# Also look at the raw distances (before conversion)
res = image_db._collection.query(
    query_embeddings=[embed_text_siglip(["fresh sushi and minimalist presentation"])[0].tolist()],
    n_results=5,
    include=["distances", "documents", "metadatas"]
)
print("\nRaw distances from Chroma:")
print(res["distances"][0])
print("\nDocuments:")
print(res["documents"][0])

Image collection metadata:
None

Raw distances from Chroma:
[1.8901782035827637, 1.9024513959884644, 1.910590410232544, 1.9184927940368652, 1.92277193069458]

Documents:
['Lemon Ricotta Pancakes', 'Roasted Tomato and Basil Pasta', 'Garlic Butter Shrimp', 'Distance-Glazed Salmon with Dill', 'Lentil Dal with Cumin and Turmeric']


### 📸 Screenshot Submission Requirement

At the end of this lab, take a screenshot of the final code cell and its output, and name it  **M2L3_multimodal_fusion_results.jpg**.

Your screenshot must clearly show:
- Multimodal fusion results (ranked outputs)
- Fused ranking results for multiple weight configurations (for example, w_text=0.8 and w_text=0.3)
- The final completion message:  
  **"Multimodal Similarity Fusion and Retrieval Ranking COMPLETE"**

This screenshot will later serve as a component for the assessment.


## Conclusion

In this lab, you implemented a production-style multimodal retrieval pipeline that retrieves from text and image vector indexes, normalizes cross-modal similarity scores, and fuses results into a unified ranked list with optional metadata constraints.

This fusion-based reranking pattern is a core building block for high-precision multimodal RAG systems.


## Authors


[Zikai Dou](https://author.skills.network/instructors/zikai_dou)


Copyright © IBM Corporation. All rights reserved.
